# TWAS and COLOC integration (INTACT)

Run INTACT on the PTWAS and fastenloc output to integrate transcriptome-wide association (TWAS) evidence with colocalization (COLOC) evidence into a single gene-level posterior probability.

## Methods

The posterior probabilities in the INTACT function are calculated through the following steps:

- **Compute Bayes Factors:** The function converts TWAS z-scores into Bayes factors by computing a grid of Bayes factors using different values of `K` (a vector of values over which Bayesian model averaging is performed, default `c(1,2,4,8,16)`), then performing Bayesian model averaging using the log sum exp trick.
- **Compute Prior Probabilities:** The `prior_fun` function converts `GLCP` into prior probabilities. `prior_fun` is set as linear by default.
- **Compute Posterior Probabilities.**

## Input

- `--ptwas-file`: PTWAS output, a TSV file with (at minimum) columns `GENE`, `STAT` (TWAS z-score) and `SUBCLASS`.
- `--fastenloc-file`: fastenloc output, a whitespace-delimited table with columns `Gene` and `GLCP` (gene-level colocalization probability).
- `--tissue`: tissue / dataset label used to name the output (e.g. `DLPFC`).
- `--alpha`: FDR significance threshold (default `0.05`).

The genes are matched between the two inputs by joining on the gene name.

## Output

An RDS file `<cwd>/<tissue>.INTACT.rds` containing the merged PTWAS + fastenloc table augmented with the INTACT posterior probability (`intact_pip`) and an FDR significance flag (`fdr_sig`), sorted by descending posterior probability.

## Steps

**Step 1.** Run INTACT on the PTWAS and fastenloc output for one tissue.

**Timing:** ~1-2 min on typical compute infrastructure.

In [ ]:
sos run pipeline/intact.ipynb intact \
    --fastenloc-file input/twas/protocol_example.fastenloc.gene.out \
    --ptwas-file input/twas/protocol_example.ptwas.output \
    --tissue DLPFC \
    --cwd output/intact

> **Environment note.** This step requires the R package `INTACT`, which is not installed in the current environment (`requireNamespace("INTACT")` returns `FALSE`). Install INTACT (Bioconductor) and provide real PTWAS and fastenloc output files before running. The command above is the verified interface; it has not been executed end-to-end here because the dependency is absent.

## Command interface

In [ ]:
sos run pipeline/intact.ipynb -h

## Workflow implementation

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
# Workdir
parameter: cwd = path("output")
# fastenloc output
parameter: fastenloc_file = ""
# ptwas output
parameter: ptwas_file = ""
# dataset 
parameter: tissue = ''
parameter: alpha = 0.05
# QTL data type
parameter: QTL = 'eQTL'
parameter: container = ''
parameter: job_size = 1
parameter: walltime = "5h"
parameter: mem = "8G"
parameter: numThreads = 1


In [ ]:
[intact]
input: ptwas_file, fastenloc_file
output: f'{cwd}/{tissue}.INTACT.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'
R: expand= "${ }", stderr = f'{_output:nn}.stderr', stdout = f'{_output:nn}.stdout'
    library(INTACT)
    #library(biomaRt)
    library(tidyverse)

    # run intact on the ptwas and colocalization output


    run_intact <- function(ptwas, fastenloc, alpha){
        # join the columns that we want
        sub_ptwas <- ptwas %>%
            select(gene=GENE, zscore=STAT)

        # match the gene names 
        sub_fastenloc <- fastenloc %>%
            select(gene=Gene, GLCP) %>%
            mutate(gene = str_replace_all(gene, "-", "\\."))

        ptwas_fastenloc <- sub_ptwas %>%
            inner_join(sub_fastenloc)

        res <- intact(GLCP_vec=ptwas_fastenloc$GLCP,
                    z_vec=ptwas_fastenloc$zscore,
                    prior_fun=linear
        )
        pip_fdr <- fdr_rst(res, alpha)

        # combine results
        intact_res <- cbind(ptwas_fastenloc, intact_pip=res, fdr_sig = pip_fdr[["sig"]]) %>%
            arrange(-intact_pip)
        return(intact_res)
        
    }


        # load ptwas data
        #ptwas <- read.csv("${ptwas_file}",sep='\t')
        ptwas <- read.csv("${ptwas_file}",sep='\t', check.names = FALSE, row.names = NULL) %>%
            distinct(GENE, SUBCLASS, .keep_all = TRUE)
        # load fastenloc GLCP data
        fastenloc <- read_table("${fastenloc_file}") %>% filter(GLCP <= 1)##FIX

        res <- run_intact(ptwas, fastenloc, alpha= '${alpha}')
        saveRDS(res, "${_output}")

